In [211]:
import numpy as np 
import pandas as pd
from linear_regression import LinearRegression

Train and Evaluate that Linear Regression model on the my dataset

In [212]:
def organize_data(data):
  X = data[["gia_ban","gia_goc","so_tien_da_giam","danh_gia_trung_binh"]].values
  Y = data["luot_ban"].values
  return X, Y
if __name__ == "__main__":
        data = pd.read_csv('tiki_dienthoaimaytinhbang.csv')
        data = data.sample(frac = 1, ignore_index=True)
        train_data = data.iloc[:90]
        test_data = data.iloc[90:]
        X_train, Y_train = organize_data (train_data)
        X_test, Y_test = organize_data(test_data)
        model = LinearRegression()
        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_test)
        result = model.evalute(Y_test, Y_pred)
        print(result)

1184.6329400149336


Standardize the data so that their mean is $0$ and their variance is $1$.

In [213]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train_standardize = (X_train - mean) / std
X_test_standardize = (X_test - mean) / std
model_standardize = LinearRegression()
model_standardize.fit(X_train_standardize, Y_train)
Y_pred_standardize = model_standardize.predict(X_test_standardize)
result_standardize = model_standardize.evalute(Y_test, Y_pred_standardize)
print(result_standardize)

1184.6066624383113


Kết quả của mô hình standardize đã chuẩn hoá có rmse có lúc thấp hơn, có lúc cao hơn mô hình chưa chuẩn hoá chạy trên bộ dữ liệu tiki_dienthoaimaytinhbang, rmse gần như không đổi 

Perform feature engineering to remove columns that are linearly dependent (colinear) with other columns in the dataset.

In [214]:
print(data[["gia_ban","gia_goc","so_tien_da_giam","danh_gia_trung_binh"]].corr())
def organize_data_original(data):
  X = data[["gia_ban","gia_goc","so_tien_da_giam","danh_gia_trung_binh"]].values
  Y = data["luot_ban"].values
  return X, Y
def organize_data_preprocessed(data):
  X = data[["gia_ban","so_tien_da_giam", "danh_gia_trung_binh"]].values
  Y = data["luot_ban"].values
  return X, Y
X_train_original, Y_train_original = organize_data_original(train_data)
X_test_original, Y_test_original = organize_data_original(test_data)
X_train_preprocessed, Y_train_preprocessed = organize_data_preprocessed(train_data)
X_test_preprocessed, Y_test_preprocessed = organize_data_preprocessed(test_data)

                      gia_ban   gia_goc  so_tien_da_giam  danh_gia_trung_binh
gia_ban              1.000000  0.994564         0.153705             0.076565
gia_goc              0.994564  1.000000         0.255757             0.054442
so_tien_da_giam      0.153705  0.255757         1.000000            -0.194225
danh_gia_trung_binh  0.076565  0.054442        -0.194225             1.000000


Evaluate the Linear Regression model when beeing trained on the original data and preprocessed data.

In [215]:
model_original = LinearRegression()
model_original.fit(X_train_original, Y_train_original)
Y_pred_original = model_original.predict(X_test_original)
result_original = model_original.evalute(Y_test_original, Y_pred_original)
print("Gốc:", result_original)
model_preprocessed = LinearRegression()
model_preprocessed.fit(X_train_preprocessed, Y_train_preprocessed)
Y_pred_preprocessed = model_preprocessed.predict(X_test_preprocessed)
result_preprocessed = model_preprocessed.evalute(Y_test_preprocessed, Y_pred_preprocessed)
print("Đã xử lí: ", result_preprocessed)

Gốc: 1184.6329400149336
Đã xử lí:  1138.4595349402211


Từ ma trận tương quan thấy cột gia_ban và gia_goc có độ tương quan rất cao và gia_goc = gia_ban + so_tien_da_giam nên mình bỏ bớt cột gia_goc để tránh đa cộng tuyến. Khi chạy lại thì rmse của dữ liệu gốc và dữ liệu đã xử lí gần như ngang nhau nhưng bỏ đi giúp ma trận X^T X ổn định hơn

Implement the Linear Regression model using Machine Learning libraries (Scikit Learn or Skorch).

In [216]:
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
from sklearn.metrics import mean_squared_error

model_sklearn = SklearnLinearRegression()
model_sklearn.fit(X_train, Y_train)
Y_pred_sklearn = model_sklearn.predict(X_test)
result_sklearn = np.sqrt(mean_squared_error(Y_test, Y_pred_sklearn))
print(result_sklearn)

1155.7752114930333


Compare the results of Linear Regression model constructed manually and from Machine Learning libraries

In [217]:
from sklearn.linear_model import LinearRegression as SklearnLinearRegression

model_sklearn = SklearnLinearRegression()
model_sklearn.fit(X_train, Y_train)
Y_pred_sklearn = model_sklearn.predict(X_test)
result_sklearn = model.evalute(Y_test, Y_pred_sklearn)
print("Sklearn:", result_sklearn)
print("Tự viết :", result)

Sklearn: 1155.7752114930333
Tự viết : 1184.6329400149336


Kết quả rmse của sklearn và code tự viết chênh nhau không đáng kể. Khác biệt một chút là do sklearn mặc định có thêm hệ số tự do (intercept) còn code tự viết thì mình chưa ghép thêm cột 1 vào ma trận X.